# GLD Signal Construction

**Author:** Jan  
**Last updated:** 2026-05-23

Translates the validated contemporaneous relationship from notebook 02 (GLD return negatively related to Δ 10Y real yield) into a target GLD weight time series. Pipeline: smooth Δ DFII10, standardize to a z-score against a trailing window, negate, map to a tilt around a baseline weight via tanh, clip to [0, 1].

Honest caveat upfront: monthly Δ DFII10 is essentially white noise (lag-1 autocorrelation ≈ 0.02), so this is a state-tracking signal, not a forward predictor. The validated regression establishes the *direction* of the tilt; the magnitude of the edge is quantified by the KPI panels in notebooks 15 (training + 2025 validation) and 16 (2026 test).

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import linregress

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TRAIN_END
from src.data import load_etf_prices, load_fred_series

pd.options.display.float_format = "{:.4f}".format
np.random.seed(0)

## Data

DFII10 (10Y real yield from TIPS) and GLD adjusted close, aligned to month-end last values. Inception of the overlap is 2004-11-18 (GLD launch).

**Filter to training period.** All pipeline visualisations here are restricted to `<= TRAIN_END` (2024-12-31). The production pipeline in `src/fundamental.py` can be called for any date, including the live test period; this notebook just shows the mechanism on training data.

In [ ]:
real_yield = load_fred_series("DFII10", start="2004-11-18", end=str(TRAIN_END.date())).resample("ME").last()
gld_full = load_etf_prices("GLD", start="2004-11-18")["GLD"]
gld = gld_full[gld_full.index <= TRAIN_END].resample("ME").last()
gld_return = gld.pct_change()

## Signal construction

Pipeline (all on monthly frequency):

1. **Feature**: Δ DFII10, the monthly change in real yield (percentage points).
2. **Smooth**: 3-month rolling mean of the feature, to reduce single-month noise.
3. **Standardize**: z-score against a trailing 24-month mean and std, so the signal is dimensionless and self-adapts to regime shifts in real-yield volatility.
4. **Negate**: multiply by -1, so a *high* signal means *falling* real yields (bullish for GLD per the validated regression).
5. **Saturate**: pass through `tanh` so extreme readings don't dominate; output is in [-1, +1].
6. **Tilt**: scale by `MAX_TILT` and add to `BASELINE_WEIGHT`, giving a target weight in `[BASELINE - MAX_TILT, BASELINE + MAX_TILT]`.
7. **Clip**: enforce [0, 1] as a safety net (the long-only constraint).

Parameters chosen on prior belief, not tuned. Sensitivity analysis is in the next-steps list.

In [ ]:
SMOOTH_WINDOW = 3      # months
LOOKBACK = 24          # months for z-score normalization
BASELINE_WEIGHT = 0.25 # equal-weight baseline across the 4 ETFs
MAX_TILT = 0.15        # max deviation from baseline, so weight stays in [0.10, 0.40]

In [ ]:
delta = real_yield.diff()
delta_smoothed = delta.rolling(SMOOTH_WINDOW).mean()
z_mean = delta_smoothed.rolling(LOOKBACK).mean()
z_std = delta_smoothed.rolling(LOOKBACK).std()
z = (delta_smoothed - z_mean) / z_std

signal = -z
tilt = MAX_TILT * np.tanh(signal)
gld_weight = (BASELINE_WEIGHT + tilt).clip(0, 1).rename("gld_weight")

pipeline = pd.DataFrame({
    "DFII10": real_yield,
    "delta": delta,
    "delta_smoothed": delta_smoothed,
    "z_score": z,
    "signal": signal,
    "tilt": tilt,
    "weight": gld_weight,
})
pipeline.tail()

## Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(signal.index, signal, color="steelblue", linewidth=1)
axes[0].axhline(0, color="gray", alpha=0.5, linewidth=0.8)
axes[0].set_ylabel("Signal")
axes[0].set_title("Signal: -z(smoothed Δ DFII10). Positive = falling real yields = bullish GLD.")
axes[0].grid(alpha=0.3)

axes[1].plot(gld_weight.index, gld_weight, color="darkorange", linewidth=1.2)
axes[1].axhline(BASELINE_WEIGHT, color="gray", alpha=0.6, linestyle="--", label=f"baseline = {BASELINE_WEIGHT:.2f}")
axes[1].axhline(BASELINE_WEIGHT + MAX_TILT, color="green", alpha=0.3, linestyle=":", label="bullish cap")
axes[1].axhline(BASELINE_WEIGHT - MAX_TILT, color="red", alpha=0.3, linestyle=":", label="bearish cap")
axes[1].set_ylabel("GLD target weight")
axes[1].set_title("GLD target weight")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)

axes[2].plot(gld.index, gld, color="goldenrod", linewidth=1)
axes[2].set_ylabel("GLD price ($)")
axes[2].set_title("GLD price for reference")
axes[2].set_xlabel("")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(gld_weight.dropna(), bins=30, color="darkorange", alpha=0.7, edgecolor="white")
axes[0].axvline(gld_weight.mean(), color="black", linestyle="--", linewidth=1, label=f"mean = {gld_weight.mean():.3f}")
axes[0].axvline(BASELINE_WEIGHT, color="gray", linestyle=":", linewidth=1, label=f"baseline = {BASELINE_WEIGHT:.2f}")
axes[0].set_xlabel("GLD target weight")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Weight distribution")
axes[0].legend()
axes[0].grid(alpha=0.3)

monthly_changes_pp = gld_weight.diff().abs().dropna() * 100
axes[1].hist(monthly_changes_pp, bins=30, color="steelblue", alpha=0.7, edgecolor="white")
axes[1].axvline(monthly_changes_pp.mean(), color="black", linestyle="--", linewidth=1,
                label=f"mean = {monthly_changes_pp.mean():.2f}pp")
axes[1].axvline(25, color="red", linestyle="--", linewidth=1, label="25pp combiner cap")
axes[1].set_xlabel("Month-over-month |Δ weight| (pp)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Monthly turnover")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
diagnostic = pd.DataFrame({
    "weight": gld_weight,
    "gld_ret_next": gld_return.shift(-1),
}).dropna()

regression = linregress(diagnostic["weight"], diagnostic["gld_ret_next"])
print("Weight as predictor of next-month GLD return:")
print(f"  Slope:    {regression.slope:+.4f}")
print(f"  R-squared:{regression.rvalue**2:.4f}")
print(f"  t-stat:   {regression.slope/regression.stderr:+.2f}")
print(f"  p-value:  {regression.pvalue:.2e}")
print(f"  N:        {len(diagnostic)}")
print()

q25, q75 = diagnostic["weight"].quantile([0.25, 0.75])
top = diagnostic[diagnostic["weight"] >= q75]["gld_ret_next"]
mid = diagnostic[(diagnostic["weight"] > q25) & (diagnostic["weight"] < q75)]["gld_ret_next"]
bot = diagnostic[diagnostic["weight"] <= q25]["gld_ret_next"]

print("Avg next-month GLD return by weight quartile (state-tracking edge):")
print(f"  Top quartile (bullish, weight >= {q75:.3f}): {top.mean()*100:+.2f}% / month  (n={len(top)})")
print(f"  Mid 50%:                                      {mid.mean()*100:+.2f}% / month  (n={len(mid)})")
print(f"  Bot quartile (bearish, weight <= {q25:.3f}): {bot.mean()*100:+.2f}% / month  (n={len(bot)})")
print(f"  Top minus bot spread:                         {(top.mean()-bot.mean())*100:+.2f}pp / month")

## Results

Populate after running. Things to look at:

1. **Time series of signal and weight**: does the signal pick up the obvious regimes (2008 GFC, 2011 European debt crisis when gold rallied, 2013 taper tantrum when gold collapsed, 2020 COVID rally, 2022 Fed hike cycle that crushed real yields' counterpart)?
2. **Weight distribution**: is it centered near baseline (0.25) with reasonable spread, or does it sit at the caps too often?
3. **Monthly turnover**: month-over-month change should mostly be < 10pp, well under the 25pp combiner cap.
4. **Quantile spread**: top-quartile-weight months should have higher avg GLD return than bot-quartile-weight months. The spread is our honest read on the state-tracking edge (the regression t-stat will be weak by construction; the quantile spread is more robust).

## Notes / next steps

1. **Productionized in `src/fundamental.py`** via the team contract `get_weights(as_of_date) -> pd.Series` indexed `['ACWI', 'AGG', 'GLD', 'BSV']`. Lookahead-safe via `load_fred_as_of("DFII10", as_of_date)`. GLD weight from this pipeline; ACWI/AGG/BSV split the remainder equally. No macro-regime overlay: regime classification belongs to Rayane's vertical.
2. **KPIs and out-of-sample validation** live in `notebooks/15_kpis_jan.ipynb` (training + 2025 validation) and `notebooks/16_test_2026_jan.ipynb` (2026 test window).
3. **Parameter sensitivity** not pursued. Fixed values `SMOOTH_WINDOW=3`, `LOOKBACK=24`, `MAX_TILT=0.15` were chosen on prior belief and not tuned to data.
4. **Complementary GLD inputs (T10YIE, DTWEXBGS)** were tested and rejected: contemporaneous strength didn't translate to forward portfolio outperformance.